# Audit an unsupervised clustering study

Explore structure in chemical measurements without training on cultivar labels. Select k using a declared training-only silhouette comparison, then inspect held-out geometry and external-label agreement. Cluster IDs have no inherent semantic names.

Dataset: scikit-learn wine benchmark · 178 observations, 13 numeric features

Reference: https://scikit-learn.org/stable/datasets/toy_dataset.html#wine-dataset

Original ML Atlas notebook, MIT-licensed code and CC BY 4.0 explanation. Third-party data retains its own license.


## Environment

Run in Jupyter or Colab. For the scikit-learn projects, install scikit-learn >=1.4 and its dependencies in your own environment. No GPU is needed.


## Plan

1. Hold out 25% of observations. Keep labels out of preprocessing and k selection.
2. Fit StandardScaler on training features only.
3. Compare k=2 through 5 using training silhouette, with multiple center initializations.
4. Use the selected centers to assign held-out observations and report held-out silhouette when defined.
5. Report adjusted Rand agreement with held-out labels as an external audit; explain why it does not validate every possible use of the clusters.


In [ ]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

data = load_wine()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=.25, random_state=42)
scaler = StandardScaler().fit(X_train)
train, test = scaler.transform(X_train), scaler.transform(X_test)
candidates = []
for k in range(2, 6):
    model = KMeans(n_clusters=k, n_init=10, random_state=42).fit(train)
    score = silhouette_score(train, model.labels_)
    candidates.append((score, k, model))
    print('Training k/silhouette:', k, round(score, 4))
_, chosen_k, selected = max(candidates, key=lambda item: item[0])
labels = selected.predict(test)
print('Selected k:', chosen_k)
print('Held-out counts:', np.bincount(labels, minlength=chosen_k).tolist())
if 1 < len(set(labels)) < len(labels):
    print('Held-out silhouette:', round(silhouette_score(test, labels), 4))
print('External-label adjusted Rand audit:', round(adjusted_rand_score(y_test, labels), 4))
print('Centers in original units:', scaler.inverse_transform(selected.cluster_centers_).round(2))


## Review the result

- [ ] Labels do not select preprocessing or cluster count.
- [ ] Evaluation data uses training scaling without refitting.
- [ ] The k-selection table and random seed are reported.
- [ ] A geometric score is distinguished from external-label agreement.
- [ ] The report checks multiple seeds and avoids treating cluster numbers as natural categories.


## Extend it

Compare the selected solution across at least five seeds and report adjusted Rand agreement between their assignments. Investigate how raw-unit versus standardized geometry changes the result.

Record what changed, why, and how you evaluated it.
